# Function 3 - Week 1: what should the first query teach us?

**Taper-waper: start by reducing ignorance, finish by exploiting what we have learned.**

There are 15 supplied observations in 3 dimensions and thirteen additional queries available. We want the first answer to improve later decisions, while recognising that a model can be precise about the wrong explanation.

In [ ]:
from pathlib import Path
import sys
stage = next(p / "stage-2-bbo" for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "stage-2-bbo/src").is_dir())
sys.path.insert(0, str(stage / "src"))
from initial_exploration import load_initial_data, preview, plot_ordered_outputs
from multifunction_report import *
data = load_initial_data(stage, 3)
figures = stage.parent / ".runtime/function-3-risk"
figures.mkdir(parents=True, exist_ok=True)
from risk_refinement import ensure_refined_results
risk = ensure_refined_results(data, 3, figures)

## Give risk a decision and a unit

Let G be the finite set of candidate locations, μ the current fitted mean, and x̂ the location with the largest fitted mean. We measure **expected simple regret**: the output we expect to forgo by recommending x̂ instead of the best location in G.

**R(D) = E[max<sub>x ∈ G</sub> f(x) − f(x̂) | D]**

For a possible query q, the **knowledge gradient** is the expected improvement in the best fitted mean after seeing its answer:

**KG(q) = E<sub>Yq | D</sub>[max<sub>x ∈ G</sub> μ<sub>D+(q,Yq)</sub>(x)] − max<sub>x ∈ G</sub> μ<sub>D</sub>(x).**

With model parameters held fixed, this equals the expected reduction in R. Values are in **supplied output units**, not probabilities or percentages. A disappointing answer can still be valuable if it changes where we would look next. [Knowledge-gradient definition](https://botorch.org/docs/v0.16.0/tutorials/one_shot_kg).

The final recommendation may be any location in G, including an unqueried one. This objective differs from maximising the portal's best noisy recorded score. It does not require inventing a threshold for a “good” result.

## Three explanations to put under pressure

**Additive:** each input has its own possibly nonlinear effect, and those effects add. This excludes interactions.

**Local:** a Matérn 3/2 Gaussian process measures similarity using one shared length scale across all inputs. This allows local features and interactions.

**Flexible:** the same type of process has a separate length scale for each input. Some inputs can matter over much shorter distances than others.

All observations and original output values are retained. Internal standardisation is reversed before risk values are reported. The main assumed noise standard deviation is **10% of the supplied outputs' standard deviation**; this is a sensitivity setting, not an estimate of measurement noise. Smaller and larger settings follow. [Gaussian-process fitting and kernels](https://scikit-learn.org/stable/modules/gaussian_process.html).

In [ ]:
fit_table(risk)

In [ ]:
fit_findings(risk)

The held-out checks refit each model with one observation removed, including its standardisation and kernel parameters. Lower error is better. The training-mean baseline predicts the held-out value using the mean of the remaining values. This small-sample comparison does not provide a reliable model probability.

Kernel length scales are fitted within 0.03–2.0 input units, with three seeded restarts. The amplitudes below refer to internally standardised outputs. Boundary warnings indicate a fit pressing against those assumptions; they are not proof that a coordinate is irrelevant.

In [ ]:
kernel_table(risk)

## How much could one answer improve the decision?

Current risk is estimated using **4,096 joint posterior function draws per model**. Each query's knowledge gradient is integrated analytically over its possible Gaussian observation on the finite pool. No course function is evaluated.

In [ ]:
current_risk_table(risk)

In [ ]:
plot_query_values(risk, save_to=figures / "query-values.png")

In [ ]:
candidate_table(risk)

All three KG columns show expected shortfall reduction from **one** answer; larger is better. The orange bars mark the final proposed input. Coverage gain is the fractional reduction in mean nearest-observation distance across the full domain, checked with 65,536 space-filling integration points. It is shown for comparison and is not blended into KG. Coordinates in this table are shortened for reading; the final portal string below retains six decimals.

In [ ]:
risk_findings(risk)

## Knowledge gradient through the cube

Each panel fixes x3 and shows the value of querying the resulting **three-coordinate** point. These are actual scored candidates on three slices of the nine-point-per-axis grid. The omitted direction is fixed, not averaged away. All nine panels share one colour scale.

The selected point need not lie on one of these three planes. Its exact coordinates are recorded in the candidate table.

In [ ]:
plot_cube_kg(risk, save_to=figures / "cube-knowledge-gradient.png")

## Search resolution and noise

Up to three dimensions, the global candidate set uses a Cartesian grid. Above three, it uses a scrambled Sobol sample, all corners, and local samples and one-coordinate perturbations around the best three observations. The larger pool increases the global and local samples and changes their seed. These numerical searches are not proofs of a continuous global optimum.

The main bounded pool is augmented with each model's larger-pool winner. The table compares individual model preferences before and after expansion. The final check also recomputes **all three current risks** on the larger pool; it does not assume that a larger pool leaves the best final decision unchanged.

In [ ]:
grid_table(risk)

In [ ]:
refinement_findings(risk)

The following noise checks refit at **3% and 30% of the supplied output standard deviation**. Their final column shows the proposed query's KG relative to the best query under that particular model and noise setting, on the bounded comparison pool. Model disagreement and noise sensitivity remain distinct sources of uncertainty.

In [ ]:
noise_table(risk)

## Twelve queries remain after the first

For each explanation, simulate **64 possible functions** and noisy answers on the bounded pool. Each first-query option sees the same simulated functions and noise draws, allowing paired comparisons. Every subsequent answer updates the Gaussian posterior.

The remaining twelve queries use **updated mean + κ × latent standard deviation**, with κ decreasing linearly from 2 to 0. Repeats are allowed and fitted parameters stay fixed. The final recommendation maximises the updated mean over the pool.

This tests one declared taper policy; it does not solve an optimal thirteen-step strategy. Error bars and ± values are one simulation standard error, not uncertainty about the real unknown function.

In [ ]:
plot_rollouts(risk, save_to=figures / "risk-over-rounds.png")

In [ ]:
final_risk_table(risk)

In [ ]:
rollout_findings(risk)

## Suggested first query

The final coordinates are a reproducible proposal at portal precision, not a claim that six decimal places are scientifically resolved. The model predictions below illustrate the range of expectations; they are not measured results.

In [ ]:
recommendation_findings(risk)

In [ ]:
prediction_table(data, risk)

All results are conditional on the fitted models, assumed noise, candidate sets and fixed-parameter updates. A finite pool cannot exclude narrow peaks between its points; a Gaussian model does not establish a true upper or lower output bound. The thirteen-query simulations do not cover how model preferences or fitted parameters might change as real evidence arrives.

The taper remains a guide to the purpose of the next answer: reduce consequential uncertainty first, then refine promising regions when the evidence supports doing so. **No portal query has been submitted.**